# 🤖 Pearl.AI — Forecasting Training

Notebook de treinamento de modelos de previsão de receita para o Pearl.AI.

**O que este notebook faz:**
1. Carrega dados de receita diretamente do banco SQLite **ou** via API REST
2. Realiza EDA (análise exploratória)
3. Treina 3 modelos: **Prophet**, **XGBoost** e **LSTM** (opcional)
4. Compara com o método atual de Holt
5. Exporta forecasts em CSV e os importa de volta para o Pearl.AI via API

---

**Configuração mínima:** Python 3.10+ | `pip install -r requirements.txt`

**Para Colab:** use `pearl_ai_forecasting_COLAB.ipynb`

## ⚙️ 1. Configuração

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURAÇÃO — edite aqui antes de rodar
# ──────────────────────────────────────────────────────────────────────────────

# Modo de carregamento de dados
DATA_SOURCE = "sqlite"  # "sqlite" ou "api"

# [SQLite] Caminho para o banco do Pearl.AI (relativo a este notebook)
SQLITE_PATH = "../Peral.ai/holding-marketplace/prisma/prisma/dev.db"

# [API] URL base e token de autenticação
API_BASE_URL = "http://localhost:3000"
API_SESSION_TOKEN = ""  # Copie do cookie de sessão do navegador

# Empresa a treinar (None = todas as empresas com dados suficientes)
TARGET_COMPANY_ID = None  # ex: "cm123abc"

# Meses de forecast a gerar
FORECAST_MONTHS = 12

# Modelos a treinar
TRAIN_PROPHET  = True
TRAIN_XGBOOST  = True
TRAIN_LSTM     = False  # Requer GPU — habilite no Colab

# Importar forecasts de volta para o Pearl.AI após treinar?
PUSH_TO_PEARL  = True

# Diretório de saída
OUTPUT_DIR = "./output"

print(f"✅ Configuração carregada — Fonte: {DATA_SOURCE} | Forecast: {FORECAST_MONTHS} meses")

In [ ]:
import os, sys, json, warnings
import sqlite3
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import joblib

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_palette('husl')
Path(OUTPUT_DIR).mkdir(exist_ok=True)

PEARL_GREEN = '#8aa26a'
print(f"✅ Imports OK | Python {sys.version.split()[0]}")

## 📦 2. Carregamento de Dados

In [ ]:
def load_from_sqlite(db_path: str, company_id: str = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Carrega companies e revenue_records direto do SQLite do Prisma."""
    conn = sqlite3.connect(db_path)
    
    companies_query = "SELECT id, name, slug, sector, country, annualRevenue, monthlyRevenue, currency FROM Company"
    if company_id:
        companies_query += f" WHERE id = '{company_id}'"
    
    companies = pd.read_sql_query(companies_query, conn)
    
    revenue_query = """
        SELECT rr.id, rr.companyId, rr.year, rr.month, rr.amount, rr.currency, rr.type, rr.createdAt,
               c.name as companyName, c.sector
        FROM RevenueRecord rr
        JOIN Company c ON c.id = rr.companyId
        WHERE rr.month IS NOT NULL
    """
    if company_id:
        revenue_query += f" AND rr.companyId = '{company_id}'"
    revenue_query += " ORDER BY rr.companyId, rr.year, rr.month"
    
    revenue = pd.read_sql_query(revenue_query, conn)
    conn.close()
    return companies, revenue


def load_from_api(base_url: str, session_token: str, company_id: str = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Carrega dados via API REST do Pearl.AI."""
    headers = {'Cookie': f'next-auth.session-token={session_token}'}
    
    # Buscar empresas
    params = {}
    if company_id:
        params['id'] = company_id
    
    resp = requests.get(f"{base_url}/api/companies", headers=headers, params=params)
    resp.raise_for_status()
    companies_data = resp.json()
    if isinstance(companies_data, dict):
        companies_data = [companies_data]
    companies = pd.DataFrame(companies_data)
    
    # Buscar dados de receita via export endpoint
    all_revenue = []
    for _, company in companies.iterrows():
        cid = company['id']
        r = requests.get(f"{base_url}/api/ai/forecast/export/{cid}", headers=headers)
        if r.ok:
            data = r.json()
            for rec in data.get('revenueRecords', []):
                rec['companyName'] = company['name']
                rec['sector'] = company.get('sector', '')
                all_revenue.append(rec)
    
    revenue = pd.DataFrame(all_revenue)
    return companies, revenue


# ── Carregar dados ────────────────────────────────────────────────────────────
if DATA_SOURCE == "sqlite":
    print(f"📂 Conectando ao SQLite: {SQLITE_PATH}")
    companies_df, revenue_df = load_from_sqlite(SQLITE_PATH, TARGET_COMPANY_ID)
else:
    print(f"🌐 Buscando dados via API: {API_BASE_URL}")
    companies_df, revenue_df = load_from_api(API_BASE_URL, API_SESSION_TOKEN, TARGET_COMPANY_ID)

print(f"\n✅ {len(companies_df)} empresa(s) | {len(revenue_df)} registros de receita")
companies_df.head()

In [ ]:
# Preprocessamento base
revenue_df['date'] = pd.to_datetime(
    revenue_df['year'].astype(str) + '-' + revenue_df['month'].astype(str).str.zfill(2) + '-01'
)
revenue_df = revenue_df.sort_values(['companyId', 'date'])
revenue_df['amount'] = pd.to_numeric(revenue_df['amount'], errors='coerce').fillna(0)

# Filtrar apenas empresas com >= 6 meses de dados (mínimo para treinar)
counts = revenue_df.groupby('companyId')['amount'].count()
valid_ids = counts[counts >= 6].index
revenue_df = revenue_df[revenue_df['companyId'].isin(valid_ids)]
companies_df = companies_df[companies_df['id'].isin(valid_ids)]

print(f"✅ {len(valid_ids)} empresa(s) com dados suficientes para treino (>= 6 meses)")
print(f"   Período: {revenue_df['date'].min().strftime('%Y-%m')} → {revenue_df['date'].max().strftime('%Y-%m')}")
revenue_df[['companyId', 'companyName', 'date', 'amount']].tail(10)

## 📊 3. Análise Exploratória (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Pearl.AI — Análise Exploratória de Receitas', fontsize=16, color='white', y=1.02)

# 1. Receita total por empresa
ax = axes[0, 0]
totals = revenue_df.groupby('companyName')['amount'].sum().sort_values(ascending=True)
totals.plot(kind='barh', ax=ax, color=PEARL_GREEN, alpha=0.8)
ax.set_title('Receita Total por Empresa', color='white')
ax.set_xlabel('Receita Acumulada (R$)', color='white')
ax.tick_params(colors='white')

# 2. Receita consolidada ao longo do tempo
ax = axes[0, 1]
portfolio = revenue_df.groupby('date')['amount'].sum()
portfolio.plot(ax=ax, color=PEARL_GREEN, linewidth=2, marker='o', markersize=4)
ax.set_title('Receita Consolidada do Portfólio', color='white')
ax.set_ylabel('R$', color='white')
ax.tick_params(colors='white')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# 3. Distribuição mensal por empresa
ax = axes[1, 0]
for cname, group in revenue_df.groupby('companyName'):
    ax.plot(group['date'], group['amount'], label=cname, linewidth=1.5, alpha=0.8)
ax.set_title('Receita por Empresa (série temporal)', color='white')
ax.set_ylabel('R$', color='white')
ax.legend(fontsize=8, loc='upper left')
ax.tick_params(colors='white')

# 4. Variação percentual mês a mês
ax = axes[1, 1]
portfolio_pct = portfolio.pct_change().dropna() * 100
colors = [PEARL_GREEN if v >= 0 else '#ef4444' for v in portfolio_pct.values]
ax.bar(portfolio_pct.index, portfolio_pct.values, color=colors, alpha=0.8, width=20)
ax.axhline(0, color='white', linewidth=0.5, linestyle='--')
ax.set_title('Variação MoM do Portfólio (%)', color='white')
ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_portfolio.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📈 Crescimento médio mensal: {portfolio_pct.mean():.1f}% | Desvio padrão: {portfolio_pct.std():.1f}%")

## 🔮 4. Feature Engineering

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Cria features de lag e rolling para modelos ML."""
    df = df.copy().sort_values('date')
    
    # Lags
    for lag in [1, 2, 3, 6]:
        df[f'lag_{lag}'] = df['amount'].shift(lag)
    
    # Rolling stats
    df['rolling_3_mean'] = df['amount'].shift(1).rolling(3).mean()
    df['rolling_6_mean'] = df['amount'].shift(1).rolling(6).mean()
    df['rolling_3_std']  = df['amount'].shift(1).rolling(3).std()
    
    # Variação
    df['pct_change_1'] = df['amount'].pct_change(1).replace([np.inf, -np.inf], 0)
    df['pct_change_3'] = df['amount'].pct_change(3).replace([np.inf, -np.inf], 0)
    
    # Sazonalidade
    df['month_sin'] = np.sin(2 * np.pi * df['date'].dt.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['date'].dt.month / 12)
    df['quarter']   = df['date'].dt.quarter
    df['month_num'] = df['date'].dt.month
    
    # Tendência linear simples
    df['trend'] = np.arange(len(df))
    
    return df.dropna()


FEATURE_COLS = [
    'lag_1', 'lag_2', 'lag_3', 'lag_6',
    'rolling_3_mean', 'rolling_6_mean', 'rolling_3_std',
    'pct_change_1', 'pct_change_3',
    'month_sin', 'month_cos', 'quarter', 'month_num', 'trend'
]

print("✅ Feature engineering pronto. Features:", FEATURE_COLS)

## 🧮 5. Baseline — Método de Holt (atual)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def holt_forecast(series: pd.Series, periods: int = 12) -> np.ndarray:
    """Reproduz o método de Holt atual do Pearl.AI (double exponential smoothing)."""
    if len(series) < 4:
        return np.full(periods, series.mean())
    model = ExponentialSmoothing(series, trend='add', seasonal=None)
    fit   = model.fit(optimized=True)
    return fit.forecast(periods)

def evaluate_model(y_true, y_pred, name: str) -> dict:
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-6))) * 100
    print(f"  [{name}] MAE={mae:,.0f} | RMSE={rmse:,.0f} | R²={r2:.3f} | MAPE={mape:.1f}%")
    return {'model': name, 'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape}

print("✅ Baseline Holt configurado")

## 🔬 6. Treinamento — Prophet

In [ ]:
results   = {}
forecasts = {}  # {companyId: [{year, month, amount, lower, upper, method}]}

if TRAIN_PROPHET:
    try:
        from prophet import Prophet
        print("📦 Prophet importado com sucesso")
    except ImportError:
        print("❌ Prophet não instalado. Rode: pip install prophet")
        TRAIN_PROPHET = False

if TRAIN_PROPHET:
    print("\n🔮 Treinando Prophet por empresa...\n")
    prophet_models = {}
    
    for company_id in revenue_df['companyId'].unique():
        company_name = revenue_df[revenue_df['companyId'] == company_id]['companyName'].iloc[0]
        df_c = revenue_df[revenue_df['companyId'] == company_id][['date', 'amount']].copy()
        df_c.columns = ['ds', 'y']
        df_c['ds'] = pd.to_datetime(df_c['ds'])
        
        if len(df_c) < 4:
            print(f"  ⏭ {company_name}: dados insuficientes ({len(df_c)} registros)")
            continue
        
        # Split treino/teste (80/20)
        split_idx  = max(int(len(df_c) * 0.8), len(df_c) - 3)
        df_train   = df_c.iloc[:split_idx]
        df_test    = df_c.iloc[split_idx:]
        
        # Treinar Prophet
        model = Prophet(
            yearly_seasonality=True if len(df_train) >= 12 else False,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode='multiplicative',
            changepoint_prior_scale=0.05,
            interval_width=0.80
        )
        model.fit(df_train, iter=300)
        
        # Avaliar no conjunto de teste
        if len(df_test) > 0:
            future_test = model.make_future_dataframe(periods=len(df_test), freq='MS', include_history=False)
            forecast_test = model.predict(future_test)
            metrics = evaluate_model(
                df_test['y'].values,
                forecast_test['yhat'].values,
                f"Prophet-{company_name}"
            )
            results[f'prophet_{company_id}'] = metrics
        
        # Gerar forecast futuro
        future = model.make_future_dataframe(periods=FORECAST_MONTHS, freq='MS', include_history=False)
        fc     = model.predict(future)
        
        # Salvar resultados
        prophet_models[company_id] = model
        forecast_rows = []
        for _, row in fc.iterrows():
            forecast_rows.append({
                'year':       int(row['ds'].year),
                'month':      int(row['ds'].month),
                'amount':     max(0.0, float(row['yhat'])),
                'lowerBound': max(0.0, float(row['yhat_lower'])),
                'upperBound': max(0.0, float(row['yhat_upper'])),
                'confidence': 0.80,
                'method':     'prophet'
            })
        forecasts[company_id] = forecast_rows
        print(f"  ✅ {company_name}: {FORECAST_MONTHS} meses previstos")
    
    # Salvar modelos
    joblib.dump(prophet_models, f'{OUTPUT_DIR}/prophet_models.pkl')
    print(f"\n💾 Modelos Prophet salvos em {OUTPUT_DIR}/prophet_models.pkl")

## 🌲 7. Treinamento — XGBoost

In [ ]:
if TRAIN_XGBOOST:
    try:
        import xgboost as xgb
        print("📦 XGBoost importado com sucesso")
    except ImportError:
        print("❌ XGBoost não instalado. Rode: pip install xgboost")
        TRAIN_XGBOOST = False

if TRAIN_XGBOOST:
    print("\n🌲 Treinando XGBoost por empresa...\n")
    xgb_models = {}
    
    for company_id in revenue_df['companyId'].unique():
        company_name = revenue_df[revenue_df['companyId'] == company_id]['companyName'].iloc[0]
        df_c = revenue_df[revenue_df['companyId'] == company_id][['date', 'amount']].copy()
        df_c = build_features(df_c)
        
        if len(df_c) < 6:
            print(f"  ⏭ {company_name}: dados insuficientes após features")
            continue
        
        split_idx = max(int(len(df_c) * 0.8), len(df_c) - 3)
        X_train = df_c[FEATURE_COLS].iloc[:split_idx]
        y_train = df_c['amount'].iloc[:split_idx]
        X_test  = df_c[FEATURE_COLS].iloc[split_idx:]
        y_test  = df_c['amount'].iloc[split_idx:]
        
        model = xgb.XGBRegressor(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0
        )
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        
        if len(y_test) > 0:
            preds = model.predict(X_test)
            metrics = evaluate_model(y_test.values, preds, f"XGBoost-{company_name}")
            results[f'xgboost_{company_id}'] = metrics
        
        # Forecast iterativo (rollout)
        last_row   = df_c.iloc[-1].copy()
        last_date  = df_c['date'].iloc[-1]
        history    = list(df_c['amount'].values)
        xgb_forecasts = []
        
        for i in range(FORECAST_MONTHS):
            next_date = last_date + pd.DateOffset(months=i+1)
            feat = {
                'lag_1': history[-1],
                'lag_2': history[-2] if len(history) >= 2 else history[-1],
                'lag_3': history[-3] if len(history) >= 3 else history[-1],
                'lag_6': history[-6] if len(history) >= 6 else history[-1],
                'rolling_3_mean': np.mean(history[-3:]),
                'rolling_6_mean': np.mean(history[-6:]),
                'rolling_3_std':  np.std(history[-3:]) if len(history) >= 3 else 0,
                'pct_change_1':   (history[-1] - history[-2]) / (history[-2] + 1e-6) if len(history) >= 2 else 0,
                'pct_change_3':   (history[-1] - history[-3]) / (history[-3] + 1e-6) if len(history) >= 3 else 0,
                'month_sin':  np.sin(2 * np.pi * next_date.month / 12),
                'month_cos':  np.cos(2 * np.pi * next_date.month / 12),
                'quarter':    next_date.quarter,
                'month_num':  next_date.month,
                'trend':      len(df_c) + i
            }
            X_next = pd.DataFrame([feat])[FEATURE_COLS]
            pred   = max(0.0, float(model.predict(X_next)[0]))
            
            # Intervalo de confiança simples (±15%)
            margin = pred * 0.15
            xgb_forecasts.append({
                'year':       int(next_date.year),
                'month':      int(next_date.month),
                'amount':     pred,
                'lowerBound': max(0.0, pred - margin),
                'upperBound': pred + margin,
                'confidence': 0.75,
                'method':     'xgboost'
            })
            history.append(pred)
        
        xgb_models[company_id] = model
        # XGBoost sobrescreve Prophet se ativo — use o melhor modelo
        if not TRAIN_PROPHET or f'xgboost_{company_id}' not in results or \
           results.get(f'xgboost_{company_id}', {}).get('mape', 999) < \
           results.get(f'prophet_{company_id}', {}).get('mape', 999):
            forecasts[company_id] = xgb_forecasts
        
        print(f"  ✅ {company_name}: XGBoost treinado")
    
    joblib.dump(xgb_models, f'{OUTPUT_DIR}/xgboost_models.pkl')
    print(f"\n💾 Modelos XGBoost salvos em {OUTPUT_DIR}/xgboost_models.pkl")

## 🧠 8. Treinamento — LSTM (PyTorch) [opcional]

In [ ]:
if TRAIN_LSTM:
    try:
        import torch
        import torch.nn as nn
        from torch.utils.data import DataLoader, TensorDataset
        DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"📦 PyTorch {torch.__version__} | Dispositivo: {DEVICE}")
    except ImportError:
        print("❌ PyTorch não instalado. Rode: pip install torch")
        TRAIN_LSTM = False

if TRAIN_LSTM:
    class LSTMForecaster(nn.Module):
        def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
            super().__init__()
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
            self.fc   = nn.Linear(hidden_size, output_size)
        
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :])
    
    SEQ_LEN     = 6
    EPOCHS      = 100
    BATCH_SIZE  = 16
    LEARNING_RATE = 0.001
    lstm_models = {}
    scalers     = {}
    
    print(f"\n🧠 Treinando LSTM ({EPOCHS} épocas, seq_len={SEQ_LEN}, device={DEVICE})...\n")
    
    for company_id in revenue_df['companyId'].unique():
        company_name = revenue_df[revenue_df['companyId'] == company_id]['companyName'].iloc[0]
        values = revenue_df[revenue_df['companyId'] == company_id]['amount'].values.astype(float)
        
        if len(values) < SEQ_LEN + 3:
            print(f"  ⏭ {company_name}: dados insuficientes para LSTM")
            continue
        
        # Normalizar
        scaler = MinMaxScaler()
        values_norm = scaler.fit_transform(values.reshape(-1, 1)).flatten()
        scalers[company_id] = scaler
        
        # Criar sequências
        X, y = [], []
        for i in range(len(values_norm) - SEQ_LEN):
            X.append(values_norm[i:i+SEQ_LEN])
            y.append(values_norm[i+SEQ_LEN])
        X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1).to(DEVICE)
        y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1).to(DEVICE)
        
        # Split
        split = max(int(len(X) * 0.8), len(X) - 2)
        dataset_train = TensorDataset(X[:split], y[:split])
        loader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
        
        # Modelo
        model = LSTMForecaster().to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        criterion = nn.MSELoss()
        
        model.train()
        for epoch in range(EPOCHS):
            for xb, yb in loader:
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{EPOCHS} | Loss: {loss.item():.6f}")
        
        # Forecast recursivo
        model.eval()
        seed_seq = values_norm[-SEQ_LEN:]
        lstm_fc  = []
        last_date = revenue_df[revenue_df['companyId'] == company_id]['date'].iloc[-1]
        
        with torch.no_grad():
            for i in range(FORECAST_MONTHS):
                x_in  = torch.tensor(seed_seq, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(DEVICE)
                pred_n = model(x_in).item()
                pred_r = float(scaler.inverse_transform([[pred_n]])[0][0])
                margin = abs(pred_r) * 0.20
                next_date = last_date + pd.DateOffset(months=i+1)
                lstm_fc.append({
                    'year':       int(next_date.year),
                    'month':      int(next_date.month),
                    'amount':     max(0.0, pred_r),
                    'lowerBound': max(0.0, pred_r - margin),
                    'upperBound': pred_r + margin,
                    'confidence': 0.80,
                    'method':     'lstm'
                })
                seed_seq = np.append(seed_seq[1:], pred_n)
        
        lstm_models[company_id] = model
        forecasts[company_id] = lstm_fc  # LSTM tem prioridade se habilitado
        print(f"  ✅ {company_name}: LSTM treinado e forecast gerado")
    
    torch.save(lstm_models, f'{OUTPUT_DIR}/lstm_models.pt')
    joblib.dump(scalers, f'{OUTPUT_DIR}/lstm_scalers.pkl')
    print(f"\n💾 Modelos LSTM salvos")

## 📊 9. Comparativo de Modelos

In [ ]:
# Comparativo: Holt vs Modelos treinados (no conjunto de teste)
print("\n📊 Comparativo de métricas por empresa:\n")

holt_results = []
for company_id in revenue_df['companyId'].unique():
    df_c     = revenue_df[revenue_df['companyId'] == company_id].sort_values('date')
    company_name = df_c['companyName'].iloc[0]
    values   = df_c['amount'].values
    if len(values) < 4:
        continue
    split_idx = max(int(len(values) * 0.8), len(values) - 3)
    train, test = values[:split_idx], values[split_idx:]
    if len(test) == 0:
        continue
    holt_pred = holt_forecast(train, len(test))
    m = evaluate_model(test, holt_pred, f"Holt-{company_name}")
    holt_results.append(m)

print("\n🏆 Resumo de métricas (MAPE médio — menor é melhor):\n")
metrics_df = pd.DataFrame(list(results.values()) + holt_results)
if not metrics_df.empty:
    summary = metrics_df.groupby('model').agg({'mae': 'mean', 'rmse': 'mean', 'mape': 'mean', 'r2': 'mean'}).round(2)
    print(summary.sort_values('mape').to_string())
else:
    print("  (nenhum modelo testado ainda)")

## 📈 10. Visualização dos Forecasts

In [ ]:
n_companies = len(forecasts)
if n_companies == 0:
    print("Nenhum forecast gerado ainda.")
else:
    cols = min(2, n_companies)
    rows = (n_companies + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows), squeeze=False)
    fig.suptitle('Pearl.AI — Forecasts por Empresa', fontsize=14, color='white')
    
    for idx, (company_id, fc_rows) in enumerate(forecasts.items()):
        ax = axes[idx // cols][idx % cols]
        df_c = revenue_df[revenue_df['companyId'] == company_id].sort_values('date')
        company_name = df_c['companyName'].iloc[0]
        method = fc_rows[0]['method'] if fc_rows else 'N/A'
        
        # Histórico
        ax.plot(df_c['date'], df_c['amount'], label='Histórico', color=PEARL_GREEN, linewidth=2, marker='o', markersize=4)
        
        # Forecast
        fc_df = pd.DataFrame(fc_rows)
        fc_df['date'] = pd.to_datetime(fc_df['year'].astype(str) + '-' + fc_df['month'].astype(str).str.zfill(2) + '-01')
        
        ax.plot(fc_df['date'], fc_df['amount'], label=f'Forecast ({method})', color='#60a5fa', linewidth=2, linestyle='--', marker='s', markersize=4)
        ax.fill_between(fc_df['date'], fc_df['lowerBound'], fc_df['upperBound'], alpha=0.15, color='#60a5fa', label='IC 80%')
        
        # Holt (baseline)
        holt_fc = holt_forecast(df_c['amount'].values, FORECAST_MONTHS)
        ax.plot(fc_df['date'], holt_fc[:len(fc_df)], label='Holt (atual)', color='#f59e0b', linewidth=1.5, linestyle=':')
        
        ax.set_title(f"{company_name}", color='white', fontsize=11)
        ax.legend(fontsize=8)
        ax.tick_params(colors='white')
        ax.set_ylabel('Receita', color='white')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.axvline(x=df_c['date'].max(), color='rgba(255,255,255,0.3)', linestyle='-', linewidth=1)
    
    # Esconder subplots vazios
    for idx in range(n_companies, rows * cols):
        axes[idx // cols][idx % cols].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/forecasts.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n📸 Gráfico salvo em {OUTPUT_DIR}/forecasts.png")

## 💾 11. Export de Datasets (CSV)

In [ ]:
# Exportar histórico
revenue_df.to_csv(f'{OUTPUT_DIR}/pearl_revenue_history.csv', index=False)
print(f"✅ Histórico salvo: {OUTPUT_DIR}/pearl_revenue_history.csv ({len(revenue_df)} registros)")

# Exportar forecasts
all_forecasts = []
for company_id, rows in forecasts.items():
    company_name = revenue_df[revenue_df['companyId'] == company_id]['companyName'].iloc[0]
    for r in rows:
        all_forecasts.append({'companyId': company_id, 'companyName': company_name, **r})

forecast_df = pd.DataFrame(all_forecasts)
forecast_df.to_csv(f'{OUTPUT_DIR}/pearl_forecasts.csv', index=False)
print(f"✅ Forecasts salvos: {OUTPUT_DIR}/pearl_forecasts.csv ({len(forecast_df)} linhas)")

# Exportar JSON para import na API
import_payload = {company_id: rows for company_id, rows in forecasts.items()}
with open(f'{OUTPUT_DIR}/pearl_forecasts_import.json', 'w') as f:
    json.dump(import_payload, f, indent=2)
print(f"✅ Payload de import: {OUTPUT_DIR}/pearl_forecasts_import.json")

# Métricas
if results:
    pd.DataFrame(list(results.values())).to_csv(f'{OUTPUT_DIR}/model_metrics.csv', index=False)
    print(f"✅ Métricas salvas: {OUTPUT_DIR}/model_metrics.csv")

## 🚀 12. Upload para o Pearl.AI

In [ ]:
def push_forecasts_to_pearl(base_url: str, session_token: str, forecasts_dict: dict):
    """Envia os forecasts treinados para o Pearl.AI via API."""
    headers = {
        'Cookie': f'next-auth.session-token={session_token}',
        'Content-Type': 'application/json'
    }
    
    success, failed = 0, 0
    for company_id, fc_rows in forecasts_dict.items():
        payload = {'companyId': company_id, 'forecasts': fc_rows}
        try:
            resp = requests.post(
                f"{base_url}/api/ai/forecast/import",
                json=payload,
                headers=headers,
                timeout=30
            )
            if resp.ok:
                data = resp.json()
                print(f"  ✅ {company_id}: {data.get('imported', 0)} registros importados")
                success += 1
            else:
                print(f"  ❌ {company_id}: {resp.status_code} — {resp.text[:100]}")
                failed += 1
        except Exception as e:
            print(f"  ❌ {company_id}: erro de conexão — {e}")
            failed += 1
    
    print(f"\n📊 Resultado: {success} importadas | {failed} falhas")


if PUSH_TO_PEARL and forecasts:
    if not API_SESSION_TOKEN:
        print("⚠️  Configure API_SESSION_TOKEN na célula de configuração para fazer o upload.")
        print("    Você pode usar o payload JSON gerado: output/pearl_forecasts_import.json")
    else:
        print(f"🚀 Enviando {len(forecasts)} forecast(s) para {API_BASE_URL}...\n")
        push_forecasts_to_pearl(API_BASE_URL, API_SESSION_TOKEN, forecasts)
elif not forecasts:
    print("⚠️  Nenhum forecast disponível para enviar. Treine pelo menos um modelo primeiro.")
else:
    print("ℹ️  PUSH_TO_PEARL = False. Para enviar, mude para True na configuração.")

---

## ✅ Pronto!

| Arquivo gerado | Descrição |
|---|---|
| `output/pearl_revenue_history.csv` | Histórico completo de receitas |
| `output/pearl_forecasts.csv` | Forecasts em formato tabular |
| `output/pearl_forecasts_import.json` | Payload pronto para importar na API |
| `output/model_metrics.csv` | Comparativo de métricas |
| `output/prophet_models.pkl` | Modelos Prophet serializados |
| `output/xgboost_models.pkl` | Modelos XGBoost serializados |
| `output/forecasts.png` | Gráfico dos forecasts |

**Próximos passos:**
- Para retreinar: re-execute a partir da célula de carregamento
- Para Colab com GPU: use `pearl_ai_forecasting_COLAB.ipynb`
- Para importar manualmente: `POST /api/ai/forecast/import` com o JSON gerado